# LSTM-GAT with Static Graph

This notebook implements an LSTM-GAT model with **sparse attention masked by a static graph**.

The graph defines which stocks can attend to each other; attention learns
the importance weights within that structure.

**Architecture**: Shared LSTM per ticker → Sparse GAT (graph-masked attention) → Position output

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os
import sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from empyrical import (
    sharpe_ratio,
    sortino_ratio,
    max_drawdown,
    annual_return,
    annual_volatility,
    calmar_ratio,
)

import random
random.seed(40)
np.random.seed(40)

import tensorflow as tf
tf.random.set_seed(40)

print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
# Training/Test Configuration
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

VOL_TARGET = 0.15

# Graph Configuration
# Options: "cvx", "pearson", "straddle_pearson", "equity_pearson"
GRAPH_TYPE = "straddle_pearson"
ALPHA = 100    # CVX parameter (if using cvx)
BETA = 0.1     # CVX parameter (if using cvx)
TAU = 0.5      # Correlation threshold (if using pearson types)

# On-the-fly Pearson Configuration
USE_TRAINING_DATA_ONLY = True
NORMALIZE_ADJACENCY = True

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GAT Hyperparameters
HIDDEN_LAYER_SIZE = 10
GAT_UNITS = 8
ATTN_HEADS = 2
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.0005
MAX_GRADIENT_NORM = 0.01
NUM_GAT_LAYERS = 2
BATCH_SIZE = 32

print(f"Train: {TRAIN_START}-{TEST_START}")
print(f"Test:  {TEST_START}-{TEST_END}")
print(f"\nModel: LSTM-GAT with Static Graph")
print(f"  Graph type: {GRAPH_TYPE}")
if GRAPH_TYPE == "cvx":
    print(f"  Alpha: {ALPHA}, Beta: {BETA}")
else:
    print(f"  Threshold: {TAU}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}")
print(f"  GAT layers: {NUM_GAT_LAYERS}")
print(f"  Dropout: {DROPOUT_RATE}")
print(f"  Learning rate: {LEARNING_RATE}")

## 3. Helper Functions

## 3. Helper Functions

## 4. Data Loading and Preparation

In [ ]:
features_path = "data/straddle_features/features.csv"
df = pd.read_csv(features_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df)} rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Tickers: {df['ticker'].nunique()}")

In [ ]:
from gml.graph_model_inputs import GraphModelFeatures

features = GraphModelFeatures(
    df=df,
    total_time_steps=TOTAL_TIME_STEPS,
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True,
    time_features=False,
)

print("Feature generator created.")

In [ ]:
train_data = features.train
valid_data = features.valid
test_data = features.test_sliding

print("Training data:")
print(f"  inputs: {train_data['inputs'].shape}")
print(f"  outputs: {train_data['outputs'].shape}")
print(f"\nValidation data:")
print(f"  inputs: {valid_data['inputs'].shape}")
print(f"  outputs: {valid_data['outputs'].shape}")
print(f"\nTest data:")
print(f"  inputs: {test_data['inputs'].shape}")
print(f"  outputs: {test_data['outputs'].shape}")

## 5. Load Adjacency Matrix

In [ ]:
from gml.straddle_graph import load_or_compute_adjacency
from settings.default import ALL_TICKERS

adjacency = load_or_compute_adjacency(
    graph_type=GRAPH_TYPE,
    df=df if GRAPH_TYPE == "straddle_pearson" else None,
    alpha=ALPHA,
    beta=BETA,
    tau=TAU,
    tickers=ALL_TICKERS,
    train_end_year=TEST_START if (GRAPH_TYPE in ["straddle_pearson", "equity_pearson"] and USE_TRAINING_DATA_ONLY) else None,
    normalize=NORMALIZE_ADJACENCY if GRAPH_TYPE in ["straddle_pearson", "equity_pearson"] else False,
)

print(f"\nAdjacency matrix shape: {adjacency.shape}")
print(f"Non-zero entries: {(adjacency != 0).sum()}")
print(f"Value range: [{adjacency.min():.4f}, {adjacency.max():.4f}]")

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(adjacency, cmap='YlOrRd')
plt.title(f'Adjacency Matrix ({GRAPH_TYPE})')
plt.tight_layout()
plt.show()

## 6. Model Definition

In [ ]:
from gml.graph_model_gat_v2 import build_lstm_gat_sparse_model

num_tickers = train_data['inputs'].shape[1]
time_steps = train_data['inputs'].shape[2]
input_size = train_data['inputs'].shape[3]

print(f"Building LSTM-GAT Sparse model:")
print(f"  num_tickers: {num_tickers}")
print(f"  time_steps: {time_steps}")
print(f"  input_size: {input_size}")
print(f"  Graph type: {GRAPH_TYPE}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}")

model = build_lstm_gat_sparse_model(
    num_tickers=num_tickers,
    time_steps=time_steps,
    input_size=input_size,
    adjacency=adjacency,
    hidden_layer_size=HIDDEN_LAYER_SIZE,
    gat_units=GAT_UNITS,
    attn_heads=ATTN_HEADS,
    dropout_rate=DROPOUT_RATE,
    learning_rate=LEARNING_RATE,
    max_gradient_norm=MAX_GRADIENT_NORM,
    num_gat_layers=NUM_GAT_LAYERS,
)

# model.summary()

## 7. Training

In [ ]:
X_train = train_data['inputs']
y_train = train_data['outputs']
w_train = train_data['active_entries']

X_valid = valid_data['inputs']
y_valid = valid_data['outputs']
w_valid = valid_data['active_entries']

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_valid.shape[0]}")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print("="*60)
print(f"Training LSTM-GAT with Static Graph ({GRAPH_TYPE})")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}, layers: {NUM_GAT_LAYERS}")
print("="*60)

history = model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    validation_data=(X_valid, y_valid, w_valid),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Negative Sharpe)')
plt.title(f'LSTM-GAT Static ({GRAPH_TYPE}) Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")

## 8. Evaluation

In [ ]:
X_test = test_data['inputs']
predictions = model.predict(X_test)

print(f"Predictions shape: {predictions.shape}")
print(f"Test outputs shape: {test_data['outputs'].shape}")

In [ ]:
positions = predictions[:, :, -1, 0].reshape(-1)
returns = test_data['outputs'][:, :, -1, 0].reshape(-1)
captured_returns = positions * returns

dates = test_data['date'][:, :, -1, 0].reshape(-1)
identifiers = test_data['identifier'][:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({
    'time': dates,
    'identifier': identifiers,
    'position': positions,
    'returns': returns,
    'captured_returns': captured_returns,
})

results_df['time'] = pd.to_datetime(results_df['time'])
results_df = results_df[results_df['identifier'] != '0']

print(f"Results: {len(results_df)} rows")
results_df.head()

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("\n" + "="*60)
print(f"LSTM-GAT Static Graph Results ({GRAPH_TYPE})")
print("="*60)

metrics_raw = calc_metrics(daily_returns, f"LSTM-GAT Static ({GRAPH_TYPE})")
display_metrics(metrics_raw)

print(f"\nVolatility-Normalized (Target: {VOL_TARGET:.0%})")
metrics_norm, scaled_returns = calc_metrics_vol_normalized(daily_returns, f"LSTM-GAT Static ({GRAPH_TYPE})", VOL_TARGET)
display_metrics(metrics_norm)

print("\nYearly Sharpe Ratios:")
yearly_sharpes = calc_yearly_sharpes(daily_returns)
for year, sharpe_val in yearly_sharpes.items():
    print(f"  {year}: {sharpe_val:.4f}")

In [ ]:
all_daily_returns = {f'LSTM-GAT Static ({GRAPH_TYPE})': daily_returns}
plot_results(all_daily_returns, f"LSTM-GAT Static Graph ({TEST_START}-{TEST_END})")

## 9. Position Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(results_df['position'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Position')
plt.ylabel('Frequency')
plt.title('Position Distribution')
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
if GRAPH_TYPE == "cvx":
    results_dir = f"results/lstm_gat_static_cvx_{ALPHA}_{BETA}/{TEST_START}-{TEST_END}"
elif GRAPH_TYPE == "pearson":
    results_dir = f"results/lstm_gat_static_pearson_{TAU}/{TEST_START}-{TEST_END}"
elif GRAPH_TYPE == "straddle_pearson":
    norm_str = "norm" if NORMALIZE_ADJACENCY else "unnorm"
    train_str = "trainonly" if USE_TRAINING_DATA_ONLY else "alldata"
    results_dir = f"results/lstm_gat_static_straddle_{TAU}_{norm_str}_{train_str}/{TEST_START}-{TEST_END}"
elif GRAPH_TYPE == "equity_pearson":
    norm_str = "norm" if NORMALIZE_ADJACENCY else "unnorm"
    train_str = "trainonly" if USE_TRAINING_DATA_ONLY else "alldata"
    results_dir = f"results/lstm_gat_static_equity_{TAU}_{norm_str}_{train_str}/{TEST_START}-{TEST_END}"

os.makedirs(results_dir, exist_ok=True)

results_df.to_csv(os.path.join(results_dir, "captured_returns_sw.csv"), index=False)

metrics_df = pd.DataFrame([metrics_raw])
metrics_df.to_csv(os.path.join(results_dir, "metrics_raw.csv"), index=False)

metrics_norm_df = pd.DataFrame([metrics_norm])
metrics_norm_df.to_csv(os.path.join(results_dir, "metrics_vol_normalized.csv"), index=False)

yearly_df = pd.DataFrame(yearly_sharpes.items(), columns=['Year', 'Sharpe'])
yearly_df.to_csv(os.path.join(results_dir, "yearly_sharpes.csv"), index=False)

print(f"Results saved to: {results_dir}")

## 11. Summary

In [ ]:
print("="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"\nModel: LSTM-GAT with Static Graph")
print(f"  Graph type: {GRAPH_TYPE}")
if GRAPH_TYPE == "cvx":
    print(f"  Alpha: {ALPHA}, Beta: {BETA}")
else:
    print(f"  Threshold: {TAU}")
print(f"\nGAT Hyperparameters:")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}")
print(f"  GAT layers: {NUM_GAT_LAYERS}")
print(f"  Dropout: {DROPOUT_RATE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"\nTraining Period: {TRAIN_START} - {TEST_START}")
print(f"Test Period:     {TEST_START} - {TEST_END}")
print(f"\nPerformance (Raw):")
print(f"  Sharpe Ratio: {metrics_raw['Sharpe']:.3f}")
print(f"  Annual Return: {metrics_raw['E[Ret.]']:.2%}")
print(f"  Annual Volatility: {metrics_raw['Vol.']:.2%}")
print(f"  Sortino Ratio: {metrics_raw['Sortino']:.3f}")
print(f"  Max Drawdown: {metrics_raw['Max DD']:.2%}")